# Complaint Data Analysis Notebook

This notebook performs exploratory data analysis and preprocessing of consumer complaint data for the RAG Complaint Chatbot project.


In [1]:
# Import required libraries
import sys
import os

# Add src directory to path
sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 200)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
# Import our custom analyzer
from eda_preprocessing import ComplaintDataAnalyzer

# Initialize analyzer
analyzer = ComplaintDataAnalyzer()


## 1. Load Data

Load the complaint data from file. Update the path to point to your actual data file.


In [ ]:
import os
import pandas as pd
# Import the class from your script
from eda_preprocessing import ComplaintDataAnalyzer

# 1. INITIALIZE the analyzer (This fixes the NameError)
analyzer = ComplaintDataAnalyzer()

# 2. Define the path
data_path = '../data/raw/complaints.csv'

# 3. If file doesn't exist, create sample data
if not os.path.exists(data_path):
    print("Data file not found. Creating sample data...")
    from eda_preprocessing import create_sample_data
    create_sample_data()
    data_path = '../data/raw/complaints.csv'

# 4. Load the data using your updated method
# This now works because 'analyzer' is defined above
data = analyzer.load_data(data_path=data_path, sample_size=5000)

print(f"Data shape: {data.shape}")
print(f"\nFirst few rows:")
data.head()

Loading complaint data...


In [12]:
## 2. Exploratory Data Analysis

Perform comprehensive EDA to understand the data structure and quality.


SyntaxError: invalid syntax (1846316092.py, line 3)

In [ ]:
# Perform basic EDA
analyzer.basic_eda()


In [ ]:
## 3. Data Filtering

Filter the data to include only our target products and remove empty narratives.


In [ ]:
# Filter data
filtered_data = analyzer.filter_data()
print(f"\nFiltered data shape: {filtered_data.shape}")
print(f"\nProduct distribution:")
print(filtered_data['product_category'].value_counts())


In [ ]:
## 4. Data Preprocessing

Clean the text narratives and prepare for embedding.


In [ ]:
# Preprocess data
cleaned_data = analyzer.preprocess_data()
print(f"\nCleaned data shape: {cleaned_data.shape}")

# Show sample of cleaned narratives
print("\nSample of original vs cleaned narratives:")
sample = cleaned_data[['Consumer complaint narrative', 'cleaned_narrative']].head(3)
for idx, row in sample.iterrows():
    print(f"\n--- Sample {idx+1} ---")
    print(f"ORIGINAL ({len(row['Consumer complaint narrative'])} chars):")
    print(row['Consumer complaint narrative'][:200] + "...")
    print(f"\nCLEANED ({len(row['cleaned_narrative'])} chars):")
    print(row['cleaned_narrative'][:200] + "...")


In [ ]:
## 5. Additional Analysis

Perform additional analysis on the cleaned data.


In [ ]:
# Analyze word count distribution by product
fig, ax = plt.subplots(figsize=(12, 6))

products = cleaned_data['product_category'].unique()
colors = plt.cm.Set2(np.linspace(0, 1, len(products)))

for i, product in enumerate(products):
    product_data = cleaned_data[cleaned_data['product_category'] == product]['cleaned_word_count']
    ax.hist(product_data, bins=30, alpha=0.5, label=product, color=colors[i])

ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
ax.set_title('Word Count Distribution by Product Category')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../notebooks/figures/wordcount_by_product.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Create interactive plot of product distribution
product_counts = cleaned_data['product_category'].value_counts().reset_index()
product_counts.columns = ['Product', 'Count']

fig = px.pie(product_counts, values='Count', names='Product', 
             title='Complaint Distribution by Product Category',
             hole=0.3)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

# Save interactive plot
fig.write_html('../notebooks/figures/product_distribution_pie.html')


In [ ]:
# Analyze most common words
from collections import Counter
import nltk
from nltk.corpus import stopwords

# Download stopwords if not already downloaded
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

# Combine all narratives
all_text = ' '.join(cleaned_data['cleaned_narrative'].astype(str))
words = all_text.split()

# Remove stopwords
stop_words = set(stopwords.words('english'))
filtered_words = [word.lower() for word in words if word.lower() not in stop_words and len(word) > 2]

# Count word frequencies
word_counts = Counter(filtered_words)
top_words = word_counts.most_common(20)

# Plot top words
fig, ax = plt.subplots(figsize=(12, 6))
words, counts = zip(*top_words)
bars = ax.barh(range(len(words)), counts, color=plt.cm.viridis(np.linspace(0, 1, len(words))))
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)
ax.invert_yaxis()
ax.set_xlabel('Frequency')
ax.set_title('Top 20 Most Common Words (Stopwords Removed)')
plt.tight_layout()
plt.savefig('../notebooks/figures/top_words.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
## 6. Generate Report

Generate the final EDA report.


In [ ]:
# Generate comprehensive report
report_path = analyzer.generate_eda_report()
print(f"Report generated: {report_path}")

# Display summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

print(f"\nTotal complaints processed: {len(cleaned_data):,}")
print("\nBy product category:")
for product, count in cleaned_data['product_category'].value_counts().items():
    percentage = (count / len(cleaned_data)) * 100
    print(f"  {product}: {count:,} ({percentage:.1f}%)")

print(f"\nAverage words per complaint: {cleaned_data['cleaned_word_count'].mean():.1f}")
print(f"Median words per complaint: {cleaned_data['cleaned_word_count'].median():.1f}")


In [ ]:
## 7. Save Processed Data

The processed data has already been saved to `../data/processed/filtered_complaints.csv`. Let's verify:


In [ ]:
# Verify saved data
saved_data_path = '../data/processed/filtered_complaints.csv'
if os.path.exists(saved_data_path):
    saved_data = pd.read_csv(saved_data_path)
    print(f"Saved data shape: {saved_data.shape}")
    print(f"\nColumns in saved data: {list(saved_data.columns)}")
    print(f"\nFirst few rows of saved data:")
    saved_data.head()
else:
    print(f"File not found: {saved_data_path}")


In [ ]:
## Conclusion

Task 1 has been completed successfully. The data has been:
1. **Explored** to understand its structure and quality
2. **Filtered** to include only target product categories
3. **Cleaned** to prepare text for embedding
4. **Saved** for use in subsequent tasks

The processed data is now ready for Task 2: Text Chunking, Embedding, and Vector Store Indexing.
